# 03.a - Seleção de Features e Redução de Dimensionalidade

A base pós-integração continha 665 colunas. Modelos preditivos com dimensionalidade excessiva estão sujeitos a overfitting, multicolinearidade e perda de interpretabilidade clínica. Para mitigar esses riscos, foi aplicado um processo estruturado de redução de dimensionalidade em cinco etapas sequenciais:

1. **Remoção conceitual:** Variáveis com vazamento de dados (data leakage), identificadores únicos sem poder preditivo e variáveis redundantes. Ex: `motivo_saida`.
2. **Variáveis constantes:** Colunas nas quais todos os registros têm o mesmo valor.
3. **Variáveis quasi-constantes:** Colunas nas quais mais de 98% dos registros assumem o mesmo valor.
4. **Baixa correlação com o alvo:** Correlação de Pearson com `indicador_obito` inferior a |r| = 0,03.
5. **Alta ausência:** Proporção de ausência superior a 99% e textos redundantes.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Paths
ROOT = Path("..").resolve()
DATA_PATH = ROOT / 'data' / 'processed' / 'base_modelagem.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'base_modelagem_reduzida.csv'

# Config
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
# Carregamento da base
df = pd.read_csv(DATA_PATH, low_memory=False)

# Garantir que indicador_obito seja numérico
df['indicador_obito'] = pd.to_numeric(df['indicador_obito'], errors='coerce')

print(f"Formato inicial: {df.shape[0]} linhas e {df.shape[1]} colunas.")


Formato inicial: 415367 linhas e 660 colunas.


## Etapa 1: Remoção por critério conceitual
Removendo variáveis com vazamento de dados (data leakage), identificadores únicos sem poder preditivo e variáveis redundantes. Destaque para `motivo_saida`, cujo preenchimento equivale a antecipar o próprio desfecho.

In [3]:
# Colunas que representam leakage, identificadores ou data release posterior ao evento
leakage_keywords = [
    'motivo_saida', 'data_saida', 'cid_morte', 'numero_aih',
    'numero_remessa', 'sequencial', 'cnpj_hospital', 'cep_paciente',
    'cpf_gestor', 'cnpj_mantenedora', 'sequencial_remessa', 'cep_estabelecimento',
    'cpf_cnpj_estabelecimento', 'cnes_cnpj_mantenedora', 'arquivo_origem',
    'data_internacao',
    'ap01cv07', 'ap02cv07', 'ap03cv07', 'ap04cv07', 'ap05cv07', 'ap06cv07', 'ap07cv07', 'dt_atual'
]

cols_to_drop_step1 = []
for col in df.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in leakage_keywords):
        cols_to_drop_step1.append(col)

cols_to_drop_step1 = list(set(cols_to_drop_step1))
# Garantir que indicador_obito nunca seja removido acidentalmente
if 'indicador_obito' in cols_to_drop_step1:
    cols_to_drop_step1.remove('indicador_obito')

df.drop(columns=[c for c in cols_to_drop_step1 if c in df.columns], inplace=True)
print(f"Removidas {len(cols_to_drop_step1)} colunas na Etapa 1.")
print(f"Formato atual: {df.shape}")


Removidas 24 colunas na Etapa 1.
Formato atual: (415367, 636)


## Etapa 2: Remoção de variáveis constantes
Eliminação de variáveis numéricas constantes — aquelas com valor único em toda a base. Essas não possuem capacidade discriminativa.

In [4]:
constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
df.drop(columns=constantes, inplace=True)
print(f"Removidas {len(constantes)} colunas constantes na Etapa 2.")
print(f"Formato atual: {df.shape}")

Removidas 60 colunas constantes na Etapa 2.
Formato atual: (415367, 576)


## Etapa 3: Remoção de variáveis quasi-constantes
Eliminação de variáveis quasi-constantes, definidas como colunas nas quais mais de 98% dos registros assumem o mesmo valor.

In [5]:
quasi_constantes = []
for c in df.columns:
    if c != 'indicador_obito':
        top_freq = df[c].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq > 0.98:
            quasi_constantes.append(c)

df.drop(columns=quasi_constantes, inplace=True)
print(f"Removidas {len(quasi_constantes)} colunas quasi-constantes (>98%) na Etapa 3.")
print(f"Formato atual: {df.shape}")

Removidas 133 colunas quasi-constantes (>98%) na Etapa 3.
Formato atual: (415367, 443)


## Etapa 4: Correlação com o alvo — calculada apenas no treino (NB06)

O filtro de correlação de Pearson com `indicador_obito` **não é aplicado aqui** para evitar vazamento de dados (*data leakage*): calculá-lo no dataset completo deixaria informação do conjunto de teste influenciar a seleção de variáveis.

Esta célula apenas **registra o critério** (|r| ≥ 0,03). O filtro é executado no `06_predictive_modeling.ipynb`, exclusivamente sobre o conjunto de treino (2015–2022), e o resultado é aplicado via `.reindex()` ao conjunto de teste.

In [6]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'indicador_obito' in num_cols:
    num_cols.remove('indicador_obito')

print(f'Colunas numéricas disponíveis para filtro de correlação: {len(num_cols)}')
print('Filtro |r| >= 0.03 será aplicado no NB06 após o split temporal.')
print(f'Formato atual (sem alteração): {df.shape}')


Colunas numéricas disponíveis para filtro de correlação: 399
Filtro |r| >= 0.03 será aplicado no NB06 após o split temporal.
Formato atual (sem alteração): (415367, 443)


## Etapa 5: Alta ausência e textos redundantes
Exclusão de variáveis com proporção de ausência superior a 99% e variáveis textuais redundantes/não codificadas.

In [7]:
alta_ausencia = [c for c in df.columns if df[c].isna().mean() > 0.99]
df.drop(columns=alta_ausencia, inplace=True)
print(f"Removidas {len(alta_ausencia)} colunas com ausência > 99% na Etapa 5.")

# Textos redundantes ou com cardinalidade extrema (exceto codigo_cnes, que pode ser agrupado futuramente se desejado, mas aqui será limpo caso necessário).
# Para modelagem, vamos dropar objects não convertidos
text_cols = df.select_dtypes(include=['object']).columns.tolist()
cols_text_drop = [c for c in text_cols if df[c].nunique() > 100]
if 'codigo_cnes' in cols_text_drop:
    cols_text_drop.remove('codigo_cnes') # Manter cnes para referencial, se precisar

df.drop(columns=cols_text_drop, inplace=True)
print(f"Removidas {len(cols_text_drop)} colunas de texto puras/alta cardinalidade.")

print(f"Formato atual: {df.shape}")

Removidas 0 colunas com ausência > 99% na Etapa 5.
Removidas 7 colunas de texto puras/alta cardinalidade.
Formato atual: (415367, 436)


## Salvamento da Base Reduzida

In [8]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Base reduzida salva em: {PROCESSED_PATH}")
print(f"Formato final: {df.shape[0]} linhas e {df.shape[1]} colunas.")

Base reduzida salva em: /home/carolina/Documents/TCC Documentos/TCC/data/processed/base_modelagem_reduzida.csv
Formato final: 415367 linhas e 436 colunas.
